In [ ]:
import pandas as pd
import numpy as np

# Three-Class

In [ ]:
# Load Data
df_raw_data = pd.read_csv('../datasets/three_class/raw_dataset.csv')

# Preprocessing (SNV) - same as training_pipeline1
feature_cols = ['730nm', '760nm', '810nm', '860nm', '900nm', '940nm']

df_snv = df_raw_data.copy()

def snv_row(x):
    x = x.astype(float)
    mean = x.mean()
    std = x.std(ddof=1)

    if std < 1e-8:
        return x * np.nan

    return (x - mean) / std

df_snv[feature_cols] = df_snv[feature_cols].apply(
    snv_row, axis=1, result_type='expand'
)

df_snv = df_snv.dropna()

# Silhouette Score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X_sil = df_snv[feature_cols]
y_sil = df_snv['Class']

X_sil_scaled = StandardScaler().fit_transform(X_sil)

sil_score = silhouette_score(X_sil_scaled, y_sil)
print(f"Silhouette Score: {sil_score:.4f}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X0 = df_snv[feature_cols]
y0 = df_snv['Class']

X_scaled0 = StandardScaler().fit_transform(X0)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled0)

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

print(f"PC1 explained variance: {pc1_var:.2f}%")
print(f"PC2 explained variance: {pc2_var:.2f}%")
print(f"Total PC1 + PC2: {pc1_var + pc2_var:.2f}%")

plt.figure(figsize=(8, 6))

for class_name in y0.unique():
    mask = y0 == class_name
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=class_name, alpha=0.7)

plt.xlabel(f'PC1 ({pc1_var:.2f}% variance)')
plt.ylabel(f'PC2 ({pc2_var:.2f}% variance)')
plt.title('PCA Score Plot of Three-Class Spectral Dataset')
plt.legend(title='Class')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Binary Class

In [ ]:
# Load Data
df_raw_data_bin = pd.read_csv('../datasets/binary/raw_dataset.csv')

feature_cols_bin = ['730nm', '760nm', '810nm', '860nm', '900nm', '940nm']

# Outlier Removal (Stage 1) - same as training_pipeline2
def filter_scans_stage1_bin(df, keep_ratio_fallback=0.85):
    filtered_parts = []

    for sample_id, group in df.groupby('Sample No.'):
        X = group[feature_cols_bin].values

        centroid = X.mean(axis=0)

        dists = np.linalg.norm(X - centroid, axis=1)

        med = np.median(dists)
        mad = np.median(np.abs(dists - med))

        threshold = med + 1.5 * mad

        mask = dists <= threshold

        if mask.sum() < len(group) * 0.5:
            cutoff = np.quantile(dists, keep_ratio_fallback)
            mask = dists <= cutoff

        filtered_group = group[mask]
        filtered_parts.append(filtered_group)

    return pd.concat(filtered_parts, ignore_index=True)

df_stage1_bin = filter_scans_stage1_bin(df_raw_data_bin)

# SNV
df_snv_bin = df_stage1_bin.copy()

def snv_row_bin(x):
    x = x.astype(float)
    mean = x.mean()
    std = x.std(ddof=1)

    if std < 1e-8:
        return x * np.nan

    return (x - mean) / std

df_snv_bin[feature_cols_bin] = df_snv_bin[feature_cols_bin].apply(
    snv_row_bin, axis=1, result_type='expand'
)

df_snv_bin = df_snv_bin.dropna()

# Median Aggregation (Stage 2)
def stage2_median_aggregation_bin(df):
    df_agg = df.groupby('Sample No.')[feature_cols_bin].median().reset_index()
    class_map = df.groupby('Sample No.')['Class'].first().reset_index()
    df_final = pd.merge(df_agg, class_map, on='Sample No.')
    return df_final

df_stage2_bin = stage2_median_aggregation_bin(df_snv_bin)

# Silhouette Score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X_sil_bin = df_stage2_bin[feature_cols_bin]
y_sil_bin = df_stage2_bin['Class']

X_sil_scaled_bin = StandardScaler().fit_transform(X_sil_bin)

sil_score_bin = silhouette_score(X_sil_scaled_bin, y_sil_bin)
print(f"Silhouette Score: {sil_score_bin:.4f}")

In [ ]:
X0_bin = df_stage2_bin[feature_cols_bin]
y0_bin = df_stage2_bin['Class']

X_scaled0_bin = StandardScaler().fit_transform(X0_bin)

pca_bin = PCA(n_components=2)
X_pca_bin = pca_bin.fit_transform(X_scaled0_bin)

pc1_var_bin = pca_bin.explained_variance_ratio_[0] * 100
pc2_var_bin = pca_bin.explained_variance_ratio_[1] * 100

print(f"PC1 explained variance: {pc1_var_bin:.2f}%")
print(f"PC2 explained variance: {pc2_var_bin:.2f}%")
print(f"Total PC1 + PC2: {pc1_var_bin + pc2_var_bin:.2f}%")

plt.figure(figsize=(8, 6))

for class_name in y0_bin.unique():
    mask = y0_bin == class_name
    plt.scatter(X_pca_bin[mask, 0], X_pca_bin[mask, 1], label=class_name, alpha=0.7)

plt.xlabel(f'PC1 ({pc1_var_bin:.2f}% variance)')
plt.ylabel(f'PC2 ({pc2_var_bin:.2f}% variance)')
plt.title('PCA Score Plot of Binary-Class Spectral Dataset')
plt.legend(title='Class')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()